# [5.4] Mamba State Tracking - Solutions

These cells validate the reference implementation in `solutions.py`. The lesson is not just probe fitting: it is a state-tracking evidence ladder with held-out positions, interventions, random controls, trained model organisms, and official hidden-state extraction.

<img src="../../instructions/assets/mamba_state_tracking_ladder.svg" width="820">


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter5_modern_architectures"
section = "part4_mamba_state_tracking"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

from part4_mamba_state_tracking import solutions, tests


## 1. Synthetic Tasks

<details><summary>Expected output</summary>

```text
All tests in `test_generate_parity_task_matches_cumulative_xor` passed!
All tests in `test_generate_bracket_depth_task_is_bounded_and_consistent` passed!
```

</details>

<details><summary>Help - solution idea</summary>

Parity is cumulative XOR. Bracket depth is a bounded random walk where the sampled action is overridden at boundaries before depth is updated.

</details>


In [ ]:
tests.test_generate_parity_task_matches_cumulative_xor(solutions.generate_parity_task)
tests.test_generate_bracket_depth_task_is_bounded_and_consistent(solutions.generate_bracket_depth_task)


## 2. Probe Data and Held-Out Positions

<details><summary>Expected output</summary>

```text
All tests in `test_one_hot_state_features_shape_noise_and_reference` passed!
All tests in `test_make_position_split_masks_are_ordered_and_disjoint` passed!
```

</details>

<details><summary>Help - solution idea</summary>

The hidden-state control is one-hot state identity plus optional deterministic noise. The train/test masks are early positions vs later positions.

</details>


In [ ]:
tests.test_one_hot_state_features_shape_noise_and_reference(solutions.one_hot_state_features)
tests.test_make_position_split_masks_are_ordered_and_disjoint(solutions.make_position_split)


## 3. Linear Probe

<details><summary>Expected output</summary>

```text
All tests in `test_fit_linear_probe_recovers_held_out_one_hot_states` passed!
```

</details>

<details><summary>Help - solution idea</summary>

The reference probe solves a ridge-regression normal equation with a bias column, then reports train-position and held-out-position accuracy separately.

</details>


In [ ]:
tests.test_fit_linear_probe_recovers_held_out_one_hot_states(
    solutions.fit_linear_probe,
    solutions.one_hot_state_features,
    solutions.make_position_split,
    solutions.evaluate_probe_generalization,
    solutions.probe_predictions,
)


## 4. Intervention and Classifier Surface

<details><summary>Expected output</summary>

```text
All tests in `test_probe_intervention_flips_decoded_state_with_random_control` passed!
All tests in `test_tiny_mamba_state_classifier_forward_shapes` passed!
All tests in `test_notebook_contract` passed!
```

</details>

<details><summary>Help - solution idea</summary>

Use target-minus-source probe directions for intervention, compare against matched random directions, and make the classifier return token-level logits plus optional token-level hidden states.

</details>


In [ ]:
tests.test_probe_intervention_flips_decoded_state_with_random_control(
    solutions.intervention_report,
    solutions.random_direction_control,
)
tests.test_tiny_mamba_state_classifier_forward_shapes(solutions.TinyMambaStateClassifier)
tests.test_notebook_contract(solutions.run_smoke_test)


## Signature Result

| Check | Current result | Acceptance rule |
|---|---:|---|
| Tiny Mamba short accuracy | `0.974` | at least `0.90` |
| Tiny Mamba long accuracy | `0.932` | at least `0.85` |
| Tiny Transformer long accuracy | `0.622` | Mamba beats by at least `0.20` |
| Learned intervention success rate | `0.989` | at least `0.90` |
| Random-direction target rate | `0.063` | at most `0.50` |
| Official hidden shape | `[4, 11, 768]` | exact shape |
| Peak VRAM | `0.324 GB` | below `24 GB` |

<details><summary>Interpreting the signature result</summary>

The reference implementation supports a generated-task Mamba state-tracking claim, not a pretrained-model behavioral claim. The official Mamba checkpoint is used for hidden-state extraction preflight only.

</details>

<details><summary>Help - checking the committed report</summary>

The cell below reads the committed report. Rerun `scripts/run_extension_verification_reports.py --section 5.4` when report inputs change.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    return {
        "device": gpu["device"],
        "torch": gpu["torch_version"],
        "cuda": gpu["cuda_version"],
        "tiny_mamba_long_accuracy": gpu["tiny_mamba_long_accuracy"],
        "tiny_transformer_long_accuracy": gpu["tiny_transformer_long_accuracy"],
        "learned_state_intervention_success_rate": gpu["learned_state_intervention_success_rate"],
        "learned_state_intervention_random_target_rate": gpu["learned_state_intervention_random_target_rate"],
        "official_mamba_hidden_shape": gpu["official_mamba_hidden_shape"],
        "official_mamba_fast_kernel_available": gpu["official_mamba_fast_kernel_available"],
        "peak_vram_gb": gpu["peak_vram_gb"],
    }


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


run_gpu_test()


## Limitations

This is a generated-task model-organism section. It validates probe/intervention methods and a trained tiny Mamba state-tracking organism, but it does not claim pretrained Mamba-130M solves these tasks or that the result generalizes beyond the declared generated setup.
